#Install Libraries

In [ ]:
# ============================================================
# Pneumonia Detection using Vision Transformer (ViT) + Grad-CAM
# Dataset: Chest X-Ray Images (Pneumonia) - Kaggle
# Model: google/vit-base-patch16-224 (fine-tuned)
# ============================================================

!pip install transformers -q
!pip install grad-cam -q
!pip install gradio -q
!pip install wandb -q

#Import Dependencies

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import ViTForImageClassification
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from sklearn.metrics import classification_report, roc_auc_score
from PIL import Image as PILImage
import gradio as gr

#  Mount Drive and Download Dataset

In [ ]:
# Mount Google Drive to save model checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Setup Kaggle credentials
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp /content/drive/MyDrive/kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')

# Download dataset
os.system('kaggle datasets download -d paultimothymooney/chest-xray-pneumonia')
os.system('unzip -q chest-xray-pneumonia.zip')

# Verify dataset
for folder in ['train', 'test', 'val']:
    path = f'chest_xray/{folder}'
    normal = len(os.listdir(f'{path}/NORMAL'))
    pneumonia = len(os.listdir(f'{path}/PNEUMONIA'))
    print(f"{folder} → NORMAL: {normal} | PNEUMONIA: {pneumonia}")

#Data Preparation

In [ ]:
# ViT expects 224x224 images
# Training set uses augmentation to reduce overfitting
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Test/val set — no augmentation, only resize and normalize
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder('chest_xray/train', transform=train_transforms)
val_dataset   = datasets.ImageFolder('chest_xray/val',   transform=test_transforms)
test_dataset  = datasets.ImageFolder('chest_xray/test',  transform=test_transforms)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Classes: {train_dataset.classes}")
print(f"Train size: {len(train_dataset)}")
print(f"Val size:   {len(val_dataset)}")
print(f"Test size:  {len(test_dataset)}")

# Load Pretrained ViT Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

# Load pretrained ViT and replace classification head with 2 classes
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=2,
    ignore_mismatched_sizes=True
)

model = model.to(device)
print("Model loaded successfully!")

#Train the Model

In [ ]:
# Class weights to handle imbalance (PNEUMONIA has 3x more images than NORMAL)
class_weights = torch.tensor([3.0, 1.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

EPOCHS = 10
best_val_acc = 0

for epoch in range(EPOCHS):
    # Training phase
    model.train()
    total_loss = 0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = correct / len(train_dataset)

    # Validation phase
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).logits
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc = val_correct / len(val_dataset)

    # Save best model to Google Drive
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(),
                   '/content/drive/MyDrive/vit_pneumonia_best.pth')
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} ✅ Saved")
    else:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

# Evaluate on Test Set

In [ ]:
model.eval()
all_preds  = []
all_labels = []
all_probs  = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images).logits
        probs   = torch.softmax(outputs, dim=1)[:, 1]

        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

print(classification_report(all_labels, all_preds,
      target_names=['NORMAL', 'PNEUMONIA']))
print(f"AUC-ROC: {roc_auc_score(all_labels, all_probs):.4f}")

#Grad-CAM Explainability

In [ ]:
# ViT wrapper — makes ViT output compatible with Grad-CAM
class ViTWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x).logits

# Reshape transform — converts ViT patch sequence to 2D spatial map
def reshape_transform(tensor, height=14, width=14):
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width,
                                       tensor.size(2))
    result = result.transpose(2, 3).transpose(1, 2)
    return result

wrapped_model = ViTWrapper(model)
target_layers = [model.vit.encoder.layer[-1].layernorm_before]
cam = GradCAM(model=wrapped_model, target_layers=target_layers,
              reshape_transform=reshape_transform)

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

# Visualize 4 test images with Grad-CAM heatmaps
fig, axes = plt.subplots(4, 2, figsize=(10, 20))

for i in range(4):
    image, label = test_dataset[i]
    input_tensor = image.unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output     = model(input_tensor).logits
        pred       = output.argmax(1).item()
        confidence = torch.softmax(output, dim=1).max().item()

    grayscale_cam = cam(input_tensor=input_tensor)[0]

    img_numpy = image.permute(1, 2, 0).numpy()
    img_numpy = std * img_numpy + mean
    img_numpy = np.clip(img_numpy, 0, 1).astype(np.float32)

    visualization = show_cam_on_image(img_numpy, grayscale_cam, use_rgb=True)

    axes[i, 0].imshow(img_numpy, cmap='gray')
    axes[i, 0].set_title(f"Original | True: {test_dataset.classes[label]}")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(visualization)
    axes[i, 1].set_title(
        f"Grad-CAM | Pred: {test_dataset.classes[pred]} ({confidence:.2%})")
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/gradcam_results.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Grad-CAM visualization saved to Google Drive!")

#Gradio Demo

In [ ]:
def predict(pil_image):
    img_tensor = test_transforms(pil_image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(img_tensor).logits
        probs  = torch.softmax(output, dim=1)[0]

    grayscale_cam = cam(input_tensor=img_tensor)[0]

    img_numpy = test_transforms(pil_image).permute(1, 2, 0).numpy()
    img_numpy = std * img_numpy + mean
    img_numpy = np.clip(img_numpy, 0, 1).astype(np.float32)

    heatmap     = show_cam_on_image(img_numpy, grayscale_cam, use_rgb=True)
    heatmap_pil = PILImage.fromarray(heatmap)

    label = {
        "NORMAL":    float(probs[0]),
        "PNEUMONIA": float(probs[1])
    }

    return label, heatmap_pil

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload Chest X-Ray"),
    outputs=[
        gr.Label(label="Prediction"),
        gr.Image(type="pil", label="Grad-CAM Heatmap")
    ],
    title="🫁 Pneumonia Detector",
    description="Upload a chest X-ray image to detect pneumonia with explainable AI (Grad-CAM)",
)

demo.launch(share=True)